In [1]:
import gc, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from google.colab import drive, runtime

IN_COLAB = True
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/IV Project-Mar 31, 2026')
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA, INTERIM_DATA
from src.benchmark import analytic_benchmark
from src.helper import make_run_dir
from src.tft import TFTModel
from src.fully_connected_colab import read_parquet_safe, load_split_bundle
from src.model3_utils import (
    FEATURE_SETS, TARGET, LOOKBACK,
    assign_sequences_to_splits, scale_sequences,
    SequenceDataset, detect_device, compute_batch_size,
    train_seq_model, predict_seq,
    hw_predict_aligned, save_seq_run,
    print_config, print_feature_set_summary,
)


Mounted at /content/drive


## Configuration — TFT rand_C

In [2]:
DATA_SET = 'rand_C'
NOTEBOOK_NAME = '3.5-tft-rand-C-colab'

# ── Hyperparameters ──
SEED = 42
MAX_EPOCHS = 100
PATIENCE = 25
LR_PATIENCE = 8
LR_FACTOR = 0.3
WARMUP_EPOCHS = 5
HIDDEN_DIM = 64
N_HEADS = 4
NUM_LAYERS = 1
DROPOUT = 0.1
BASE_LR = 1e-3
BASE_BATCH = 512

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)


## Load data

In [3]:
df_train, df_val, df_test = load_split_bundle(CLEAN_DATA, DATA_SET)

# Load full ordered data for sequence construction
df_full = read_parquet_safe(INTERIM_DATA / '03-data-merge-feature.parquet')

# Filter to range C dates
df_full = df_full[df_full['date'] >= '2020-03-23'].reset_index(drop=True)

print(f'Train: {len(df_train):,}  Val: {len(df_val):,}  Test: {len(df_test):,}')
print(f'Full data for sequences: {len(df_full):,} rows')


Train: 1,716,781  Val: 490,509  Test: 245,255
Full data for sequences: 2,452,545 rows


## GPU auto-detect

In [4]:
CFG = detect_device()
DEVICE = CFG['DEVICE']
AMP_DTYPE = CFG['POLICY']
USE_AMP = (AMP_DTYPE != torch.float32)


L4  |  VRAM: 24 GB  |  MAX_BATCH=2,048  |  dtype=torch.bfloat16


## Fit Hull-White benchmark (once)

In [5]:
hw = analytic_benchmark(df_train, df_val, df_test, target=TARGET)
hw_coef = hw['coef']


Analytic Benchmark
SSE = 64.9134  RMSE = 0.016269
Coefficients: a = -0.078535, b = -0.092385, c = -0.081882


## Train across feature sets

In [6]:
OUTPUT_DIR = ROOT / 'output' / NOTEBOOK_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results_by_fs = {}
df_sorted_ref = None  # will be set on first feature set

for fs_name, feature_cols in FEATURE_SETS.items():
    print_feature_set_summary(fs_name, 0, 0, 0, feature_cols)

    # Build sequences from full ordered data, assign by target row's split
    seq_data = assign_sequences_to_splits(
        df_full, df_train, df_val, df_test,
        feature_cols=feature_cols, target=TARGET, lookback=LOOKBACK,
    )

    X_tr, y_tr = seq_data['X_train'], seq_data['y_train']
    X_va, y_va = seq_data['X_val'], seq_data['y_val']
    X_te, y_te = seq_data['X_test'], seq_data['y_test']
    test_idx = seq_data['test_indices']
    df_sorted = seq_data['df_sorted']
    if df_sorted_ref is None:
        df_sorted_ref = df_sorted

    print(f'  Sequences — Train: {len(X_tr):,}  Val: {len(X_va):,}  Test: {len(X_te):,}')

    # Scale
    X_tr_sc, X_va_sc, X_te_sc, scaler = scale_sequences(X_tr, X_va, X_te)

    # DataLoaders with adaptive batch size
    BATCH_SIZE = compute_batch_size(len(X_tr), CFG['MAX_BATCH'])
    INIT_LR = BASE_LR * (BATCH_SIZE / BASE_BATCH) ** 0.5
    print_config(CFG, BATCH_SIZE, INIT_LR, len(X_tr), MAX_EPOCHS, PATIENCE, WARMUP_EPOCHS, LOOKBACK)

    train_ds = SequenceDataset(X_tr_sc, y_tr)
    val_ds = SequenceDataset(X_va_sc, y_va)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True)

    # Build model
    n_feat = len(feature_cols)
    model = TFTModel(
        n_features=n_feat,
        hidden_dim=HIDDEN_DIM,
        n_heads=N_HEADS,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        seed=SEED,
    ).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  TFT params: {n_params:,}')

    # Train
    train_result = train_seq_model(
        model, train_loader, val_loader,
        device=DEVICE, amp_dtype=AMP_DTYPE, use_amp=USE_AMP,
        max_epochs=MAX_EPOCHS, patience=PATIENCE,
        lr_patience=LR_PATIENCE, lr_factor=LR_FACTOR,
        init_lr=INIT_LR, warmup_epochs=WARMUP_EPOCHS,
        use_tqdm=True, desc=fs_name,
    )

    # Predict on test
    y_pred = predict_seq(train_result['model'], X_te_sc, DEVICE, AMP_DTYPE, USE_AMP)

    # HW SSE aligned to this feature set's test rows
    _, _, hw_sse = hw_predict_aligned(hw_coef, df_sorted, test_idx)

    from src.metrics import metrics, gain
    met = metrics(y_te, y_pred)
    g = gain(met['SSE'], hw_sse) * 100
    print(f'  {fs_name}: SSE={met["SSE"]:.4f}  RMSE={met["RMSE"]:.6f}  '
          f'Gain={g:+.2f}%  epochs={train_result["epochs"]}  '
          f'time={train_result["training_time"]:.1f}s')

    results_by_fs[fs_name] = {
        'model': train_result['model'],
        'y_pred': y_pred,
        'y_true': y_te,
        'test_indices': test_idx,
        'history': train_result['history'],
        'epochs': train_result['epochs'],
        'training_time': train_result['training_time'],
        'scaler': scaler,
    }

    # Free memory
    del X_tr, X_va, X_te, X_tr_sc, X_va_sc, X_te_sc
    del train_ds, val_ds, train_loader, val_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()



  Feature set: 4F  (4 features)
  Train: 0  Val: 0  Test: 0 sequences
  Features: delta, T, spy_ret, vix_lag
  Sequences — Train: 1,169,131  Val: 333,671  Test: 166,699
MAX_BATCH=2,048  adaptive BATCH_SIZE=2,048  INIT_LR=0.002000  n_train=1,169,131  steps/epoch~570
MAX_EPOCHS=100  PATIENCE=25  WARMUP=5 epochs  LOOKBACK=20
  TFT params: 161,337


4F:   0%|          | 0/100 epochs [00:00<?]  

  4F: SSE=4.3001  RMSE=0.005079  Gain=+75.97%  epochs=93  time=1413.9s

  Feature set: 5F  (5 features)
  Train: 0  Val: 0  Test: 0 sequences
  Features: delta, T, spy_ret, vix_lag, iv_lag
  Sequences — Train: 1,169,131  Val: 333,671  Test: 166,699
MAX_BATCH=2,048  adaptive BATCH_SIZE=2,048  INIT_LR=0.002000  n_train=1,169,131  steps/epoch~570
MAX_EPOCHS=100  PATIENCE=25  WARMUP=5 epochs  LOOKBACK=20
  TFT params: 178,961


5F:   0%|          | 0/100 epochs [00:00<?]  

  5F: SSE=0.5220  RMSE=0.001770  Gain=+97.08%  epochs=100  time=1592.1s

  Feature set: 6F  (6 features)
  Train: 0  Val: 0  Test: 0 sequences
  Features: delta, T, spy_ret, vix_lag, iv_lag, d_iv_lag
  Sequences — Train: 1,155,542  Val: 329,786  Test: 164,695
MAX_BATCH=2,048  adaptive BATCH_SIZE=2,048  INIT_LR=0.002000  n_train=1,155,542  steps/epoch~564
MAX_EPOCHS=100  PATIENCE=25  WARMUP=5 epochs  LOOKBACK=20
  TFT params: 196,717


6F:   0%|          | 0/100 epochs [00:00<?]  

  6F: SSE=0.4983  RMSE=0.001739  Gain=+97.13%  epochs=100  time=1744.7s

  Feature set: 8F  (8 features)
  Train: 0  Val: 0  Test: 0 sequences
  Features: delta, T, spy_ret, vix_lag, iv_lag, d_iv_lag, gamma, rho
  Sequences — Train: 1,155,542  Val: 329,786  Test: 164,695
MAX_BATCH=2,048  adaptive BATCH_SIZE=2,048  INIT_LR=0.002000  n_train=1,155,542  steps/epoch~564
MAX_EPOCHS=100  PATIENCE=25  WARMUP=5 epochs  LOOKBACK=20
  TFT params: 232,625


8F:   0%|          | 0/100 epochs [00:00<?]  

  8F: SSE=0.1703  RMSE=0.001017  Gain=+99.02%  epochs=100  time=2081.9s


## Save results

In [7]:
summary = save_seq_run(
    OUTPUT_DIR,
    results_by_fs=results_by_fs,
    hw_coef=hw_coef,
    df_sorted=df_sorted_ref,
)
print('\nMetrics Summary:')
summary


Saving results to: /content/drive/MyDrive/Colab Notebooks/IV Project-Mar 31, 2026/output/3.5-tft-rand-C-colab
Feature sets to save: ['4F', '5F', '6F', '8F']

✓ Saved metrics_summary.csv
✓ Saved residual_diagnostics.csv
✓ Saved gain_table.csv
✓ Saved 4 × 3 artifacts (weights, predictions, history)
✓ All results saved to /content/drive/MyDrive/Colab Notebooks/IV Project-Mar 31, 2026/output/3.5-tft-rand-C-colab

Metrics Summary:


,Model,SSE,MSE,RMSE,MAE,MeanError,MedianAE,R2,Training_time,Gain_vs_Analytic
0,Analytic (4F),17.891375,0.000107,0.010360,0.004405,0.000770,0.002271,0.100868,None,None
1,4F,4.300082,0.000026,0.005079,0.002180,-0.000325,0.001352,0.783899,1413.9s,75.97%
2,Analytic (5F),17.891375,0.000107,0.010360,0.004405,0.000770,0.002271,0.100868,None,None
3,5F,0.521975,0.000003,0.001770,0.001045,0.000046,0.000679,0.973768,1592.1s,97.08%
4,Analytic (6F),17.371010,0.000105,0.010270,0.004386,0.000775,0.002268,0.099597,None,None
5,6F,0.498316,0.000003,0.001739,0.000999,0.000056,0.000631,0.974170,1744.7s,97.13%
6,Analytic (8F),17.371010,0.000105,0.010270,0.004386,0.000775,0.002268,0.099597,None,None
7,8F,0.170259,0.000001,0.001017,0.000646,-0.000023,0.000392,0.991175,2081.9s,99.02%
8,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6832.6s,None


## Summary

In [8]:
total_time = sum(r['training_time'] for r in results_by_fs.values())
print(f'\nTFT on {DATA_SET} — total training time: {total_time / 60:.1f} min')
for fs_name, res in results_by_fs.items():
    from src.metrics import metrics, gain
    met = metrics(res['y_true'], res['y_pred'])
    _, _, hw_sse = hw_predict_aligned(hw_coef, df_sorted_ref, res['test_indices'])
    g = gain(met['SSE'], hw_sse) * 100
    print(f'  {fs_name}: SSE={met["SSE"]:.4f}  Gain={g:+.2f}%  epochs={res["epochs"]}')



TFT on rand_C — total training time: 113.9 min
  4F: SSE=4.3001  Gain=+75.97%  epochs=93
  5F: SSE=0.5220  Gain=+97.08%  epochs=100
  6F: SSE=0.4983  Gain=+97.13%  epochs=100
  8F: SSE=0.1703  Gain=+99.02%  epochs=100


## Cleanup

In [9]:
del results_by_fs, df_full, df_sorted_ref
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'Final VRAM: {(total - free) / 1e9:.2f} GB / {total / 1e9:.0f} GB')

print(f'Total training time: {total_time / 60:.1f} min')

if IN_COLAB:
    runtime.unassign()


Final VRAM: 0.35 GB / 24 GB
Total training time: 113.9 min
